# Notebook 1 — Schema Exploration

**Goal:** Understand the 17 tables, their row counts, column types, sample rows, and how they join together.

This is the most important notebook for interview prep. If you can walk through every table here and explain:
- What one row represents (the **grain**)
- Why it exists
- How it connects to adjacent tables

…you will answer schema questions confidently in any supply chain DS interview.

In [4]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

DB_URL = os.getenv('SUPPLY_CHAIN_DB_URL', '')
engine = create_engine(DB_URL)
SCHEMA = 'supply_chain'

def q(sql):
    return pd.read_sql(sql, engine)

print('Connected.')

ArgumentError: Could not parse SQLAlchemy URL from given URL string

## 1. Row counts across all tables

The first thing any data engineer or analyst does when landing in a new database: count rows. You want to immediately spot if something is missing, way too small, or suspiciously large.

In [5]:
tables = [
    'distribution_centers', 'stores', 'suppliers', 'products', 'customers',
    'calendar', 'product_supplier', 'promotions', 'weather',
    'purchase_orders', 'purchase_order_lines', 'shipments', 'shipment_events',
    'sales_transactions', 'inventory_snapshots', 'returns', 'spoilage'
]

counts = {}
for t in tables:
    n = q(f'SELECT COUNT(*) AS n FROM {SCHEMA}.{t}').iloc[0]['n']
    counts[t] = n

count_df = pd.DataFrame(list(counts.items()), columns=['table', 'row_count'])
count_df['row_count_fmt'] = count_df['row_count'].apply(lambda x: f'{x:,}')
print(count_df[['table', 'row_count_fmt']].to_string(index=False))

NameError: name 'q' is not defined

## 2. Dimension table samples

### 2a. Distribution Centers

8 rows. Every other table ultimately traces back here — stores belong to DCs, POs ship to DCs, outbound shipments originate from DCs.

In [ ]:
q(f'SELECT * FROM {SCHEMA}.distribution_centers ORDER BY dc_id')

### 2b. Stores — geographic distribution

In [ ]:
# Store format distribution
q(f"""
SELECT store_format, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM {SCHEMA}.stores
GROUP BY store_format
ORDER BY n DESC
""")

In [ ]:
# Stores per DC
q(f"""
SELECT d.dc_name, d.region, COUNT(s.store_id) AS store_count
FROM {SCHEMA}.distribution_centers d
JOIN {SCHEMA}.stores s ON s.dc_id = d.dc_id
GROUP BY d.dc_name, d.region
ORDER BY store_count DESC
""")

### 2c. Products — category breakdown

In [ ]:
q(f"""
SELECT category,
       COUNT(*) AS sku_count,
       SUM(CASE WHEN is_perishable THEN 1 ELSE 0 END) AS perishable_count,
       ROUND(AVG(unit_price), 2) AS avg_price,
       ROUND(AVG(unit_price - unit_cost), 2) AS avg_gross_margin
FROM {SCHEMA}.products
GROUP BY category
ORDER BY sku_count DESC
""")

In [ ]:
# Price range and margin by subcategory (top 15 by count)
q(f"""
SELECT subcategory, category,
       COUNT(*) AS n,
       ROUND(MIN(unit_price), 2) AS min_price,
       ROUND(MAX(unit_price), 2) AS max_price,
       ROUND(AVG(unit_price), 2) AS avg_price,
       ROUND(AVG((unit_price - unit_cost) / unit_price), 3) AS avg_margin_pct
FROM {SCHEMA}.products
GROUP BY subcategory, category
ORDER BY n DESC
LIMIT 15
""")

### 2d. Suppliers — type and reliability profile

In [ ]:
q(f"""
SELECT supplier_type,
       COUNT(*) AS n,
       ROUND(AVG(lead_time_days_avg), 2) AS avg_lead_time,
       ROUND(AVG(lead_time_days_std), 2) AS avg_lead_std,
       ROUND(AVG(otif_baseline), 3) AS avg_otif,
       ROUND(AVG(fill_rate_baseline), 3) AS avg_fill_rate,
       ROUND(AVG(disruption_prob), 4) AS avg_disruption_prob
FROM {SCHEMA}.suppliers
GROUP BY supplier_type
ORDER BY avg_lead_time
""")

### 2e. Calendar — holiday flags

In [ ]:
# All holidays in the calendar
q(f"""
SELECT date_id, holiday_name, day_name, season
FROM {SCHEMA}.calendar
WHERE is_holiday = TRUE
ORDER BY date_id
""")

## 3. Fact table samples

### 3a. Sales transactions — what does one row mean?

In [ ]:
# One row of sales + enriched with dimension data
q(f"""
SELECT
    st.transaction_id,
    st.date_id,
    s.store_name,
    s.store_format,
    p.sku,
    p.product_name,
    p.category,
    st.quantity_sold,
    st.unit_price,
    st.discount_amt,
    st.net_revenue,
    st.is_stockout_day,
    st.promotion_id
FROM {SCHEMA}.sales_transactions st
JOIN {SCHEMA}.stores s ON s.store_id = st.store_id
JOIN {SCHEMA}.products p ON p.product_id = st.product_id
LIMIT 10
""")

### 3b. Tracing a Purchase Order end-to-end

This is the most important join chain to understand:
> Supplier → Purchase Order → PO Lines → Shipment → Shipment Events

In a real interview, if someone asks "how did this product get to this store?", this is the query path.

In [ ]:
# Pick one PO and trace it
sample_po = q(f"""
SELECT po_id, po_number, supplier_id, dc_id, order_date,
       requested_delivery, actual_delivery, status, is_late, late_days
FROM {SCHEMA}.purchase_orders
WHERE status = 'Received' AND is_late = TRUE
LIMIT 1
""")
print(sample_po)
po_id = int(sample_po.iloc[0]['po_id'])

In [ ]:
# PO lines for that order
q(f"""
SELECT pol.po_line_id, p.sku, p.product_name, pol.ordered_qty,
       pol.confirmed_qty, pol.received_qty, pol.unit_cost, pol.fill_rate
FROM {SCHEMA}.purchase_order_lines pol
JOIN {SCHEMA}.products p ON p.product_id = pol.product_id
WHERE pol.po_id = {po_id}
LIMIT 10
""")

In [ ]:
# Shipment for that PO + its events
q(f"""
SELECT se.event_timestamp, se.event_type, se.location, se.delay_hours, se.notes
FROM {SCHEMA}.shipments sh
JOIN {SCHEMA}.shipment_events se ON se.shipment_id = sh.shipment_id
WHERE sh.po_id = {po_id}
ORDER BY se.event_timestamp
""")

## 4. Referential integrity checks

Run this block after every data load. All queries should return 0.

In [ ]:
integrity_checks = [
    ('FK: sales → invalid store',   f'SELECT COUNT(*) FROM {SCHEMA}.sales_transactions WHERE store_id NOT IN (SELECT store_id FROM {SCHEMA}.stores)'),
    ('FK: sales → invalid product', f'SELECT COUNT(*) FROM {SCHEMA}.sales_transactions WHERE product_id NOT IN (SELECT product_id FROM {SCHEMA}.products)'),
    ('Negative inventory',           f'SELECT COUNT(*) FROM {SCHEMA}.inventory_snapshots WHERE units_on_hand < 0'),
    ('Spoilage on non-perishable',   f"SELECT COUNT(*) FROM {SCHEMA}.spoilage sp JOIN {SCHEMA}.products p ON p.product_id = sp.product_id WHERE p.is_perishable = FALSE"),
    ('Bad promo dates',              f'SELECT COUNT(*) FROM {SCHEMA}.promotions WHERE end_date < start_date'),
    ('Return before purchase',       f'SELECT COUNT(*) FROM {SCHEMA}.returns r JOIN {SCHEMA}.sales_transactions st ON st.transaction_id = r.transaction_id WHERE r.return_date < st.date_id'),
]

all_pass = True
for desc, sql in integrity_checks:
    n = pd.read_sql(sql, engine).iloc[0,0]
    status = '✓ PASS' if n == 0 else f'✗ FAIL ({n:,} violations)'
    print(f'{status:30s}  {desc}')
    if n != 0:
        all_pass = False

print()
print('All checks passed!' if all_pass else 'Some checks FAILED — review before proceeding.')

## 5. Schema relationship map (text version)

For interview prep — be able to draw this on a whiteboard:

```
DIMENSION TABLES (the "who" and "what")
  distribution_centers ←── stores ←── (geography anchor)
  suppliers            ←── purchase_orders
  products             ←── sales_transactions, purchase_order_lines, inventory_snapshots
  customers            ←── sales_transactions
  calendar             ←── sales_transactions, weather, inventory_snapshots, ...

RELATIONSHIP TABLE
  product_supplier  (products × suppliers, with lead time + cost metadata)

FACT TABLES (the "what happened")
  sales_transactions       ← stores × products × customers × calendar × promotions
  inventory_snapshots      ← stores × products × calendar
  purchase_orders          ← suppliers × distribution_centers
  purchase_order_lines     ← purchase_orders × products
  shipments                ← purchase_orders × (supplier|dc) → (dc|store)
  shipment_events          ← shipments (log table)
  promotions               ← products × stores × calendar
  weather                  ← stores × calendar
  returns                  ← sales_transactions × products × stores
  spoilage                 ← stores × products × calendar (perishables only)
```

**The key insight:** `calendar` and `products` are **conformed dimensions** — they are shared across all fact tables. This is what makes the star schema powerful: you can join any two fact tables through a shared dimension without complex table-specific logic.

In [ ]:
# Summary stats for the interview
summary = q(f"""
SELECT
    (SELECT COUNT(*) FROM {SCHEMA}.sales_transactions)        AS total_transactions,
    (SELECT SUM(net_revenue) FROM {SCHEMA}.sales_transactions) AS total_revenue,
    (SELECT COUNT(DISTINCT store_id) FROM {SCHEMA}.sales_transactions) AS active_stores,
    (SELECT COUNT(DISTINCT product_id) FROM {SCHEMA}.sales_transactions) AS active_skus,
    (SELECT MIN(date_id) FROM {SCHEMA}.sales_transactions)    AS first_sale_date,
    (SELECT MAX(date_id) FROM {SCHEMA}.sales_transactions)    AS last_sale_date
""")
print(summary.T)